In [128]:
'''
Titanic: предсказание выживаемости пассажиров

Задача: бинарная классификация — предсказать, выжил ли пассажир (Survived).
Метрика: accuracy (Kaggle), для подбора параметров используется ROC-AUC.
Модель: Gradient Boosting (sklearn) с подбором гиперпараметров через GridSearchCV
и усреднением предсказаний по фолдам кросс-валидации.

План:
1. Загрузка и первичный осмотр данных
2. Feature engineering (Title, FamilySize, Deck, TicketPrefix)
3. Обработка пропусков
4. Кодирование категориальных признаков
5. Обучение и подбор гиперпараметров
6. Финальные предсказания и submission
'''

'\nTitanic: предсказание выживаемости пассажиров\n\nЗадача: бинарная классификация — предсказать, выжил ли пассажир (Survived).\nМетрика: accuracy (Kaggle), для подбора параметров используется ROC-AUC.\nМодель: Gradient Boosting (sklearn) с подбором гиперпараметров через GridSearchCV\nи усреднением предсказаний по фолдам кросс-валидации.\n\nПлан:\n1. Загрузка и первичный осмотр данных\n2. Feature engineering (Title, FamilySize, Deck, TicketPrefix)\n3. Обработка пропусков\n4. Кодирование категориальных признаков\n5. Обучение и подбор гиперпараметров\n6. Финальные предсказания и submission\n'

In [129]:
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [130]:
# 1. Загрузка данных

In [131]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

test_ids = test['PassengerId'].copy()

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Доля выживших в train: {train['Survived'].mean():.3f}")

Train: (891, 12), Test: (418, 11)
Доля выживших в train: 0.384


In [132]:
'''
2. Feature engineering

Создаём новые признаки на основе существующих:

Title — титул из имени (Mr, Mrs, Miss, Master, Rare)
FamilySize — размер семьи (SibSp + Parch + 1)
IsAlone — флаг одиночного путешествия
Deck — палуба из номера каюты
TicketPrefix — буквенный префикс билета

Все преобразования применяются одинаково к train и test
'''

'\n2. Feature engineering\n\nСоздаём новые признаки на основе существующих:\n\nTitle — титул из имени (Mr, Mrs, Miss, Master, Rare)\nFamilySize — размер семьи (SibSp + Parch + 1)\nIsAlone — флаг одиночного путешествия\nDeck — палуба из номера каюты\nTicketPrefix — буквенный префикс билета\n\nВсе преобразования применяются одинаково к train и test\n'

In [133]:
def feature_engineering(data):
    data = data.copy()

    # Титул из имени
    data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    data['Title'] = data['Title'].replace(rare_titles, 'Rare')
    data['Title'] = data['Title'].replace(['Mlle', 'Ms'], 'Miss')
    data['Title'] = data['Title'].replace('Mme', 'Mrs')

    # Размер семьи
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)

    # Палуба
    data['Deck'] = data['Cabin'].str[0]

    # Префикс билета
    data['TicketPrefix'] = (data['Ticket'].str.extract(r'([A-Za-z]+)', expand = False).fillna('NONE'))

    return data

train = feature_engineering(train)
test = feature_engineering(test)

In [134]:
'''
3. Обработка пропусков

Статистики (медианы, моды) считаем только на train — чтобы не было утечки
информации из тестовой выборки

Age — медиана по группам (Pclass, Sex, Title)
Fare — общая медиана
Embarked — мода
Deck — категория U для неизвестных
'''

'\n3. Обработка пропусков\n\nСтатистики (медианы, моды) считаем только на train — чтобы не было утечки\nинформации из тестовой выборки\n\nAge — медиана по группам (Pclass, Sex, Title)\nFare — общая медиана\nEmbarked — мода\nDeck — категория U для неизвестных\n'

In [135]:
# Age: медиана по группам (Pclass, Sex, Title) из train
age_medians = train.groupby(['Pclass', 'Sex', 'Title'])['Age'].median()

def fill_age(data, medians):
    data = data.copy()
    data['Age'] = data.apply(
        lambda row: row['Age'] if pd.notna(row['Age'])
        else medians.get((row['Pclass'], row['Sex'], row['Title']), np.nan),
        axis=1,
    )
    data['Age'] = data['Age'].fillna(medians.median())
    return data

train = fill_age(train, age_medians)
test = fill_age(test, age_medians)

# Fare: медиана из train
fare_median = train['Fare'].median()
train['Fare'] = train['Fare'].fillna(fare_median)
test['Fare'] = test['Fare'].fillna(fare_median)

# Embarked: мода из train
embarked_mode = train['Embarked'].mode()[0]
train['Embarked'] = train['Embarked'].fillna(embarked_mode)
test['Embarked'] = test['Embarked'].fillna(embarked_mode)

# Deck: категория U для неизвестных
train['Deck'] = train['Deck'].fillna('U')
test['Deck'] = test['Deck'].fillna('U')

In [136]:
'''
4. Удаление ненужных колонок

Cabin — слишком много пропусков, информация уже вынесена в Deck
Name — информация вынесена в Title
Ticket — оставлен только префикс
PassengerId — идентификатор, не несёт предсказательной силы

Проверяем, что после удаления пропусков не осталось.
'''

'\n4. Удаление ненужных колонок\n\nCabin — слишком много пропусков, информация уже вынесена в Deck\nName — информация вынесена в Title\nTicket — оставлен только префикс\nPassengerId — идентификатор, не несёт предсказательной силы\n\nПроверяем, что после удаления пропусков не осталось.\n'

In [137]:
drop_cols = ['Cabin', 'Name', 'Ticket', 'PassengerId']
train = train.drop(columns = drop_cols)
test = test.drop(columns = drop_cols)

assert train.isna().sum().sum() == 0
assert test.isna().sum().sum() == 0

In [138]:
'''
5. Биннинг числовых признаков

Разбиваем Age и Fare на интервалы. Границы для Fare берём по квартилям train, чтобы не подглядывать в тестовые данные
'''

'\n5. Биннинг числовых признаков\n\nРазбиваем Age и Fare на интервалы. Границы для Fare берём по квартилям train, чтобы не подглядывать в тестовые данные\n'

In [139]:
age_bins = [0, 12, 18, 35, 60, 100]
fare_bins = [-1, train['Fare'].quantile(0.25), train['Fare'].quantile(0.50), train['Fare'].quantile(0.75), train['Fare'].max() + 1]

for df in [train, test]:
    df['AgeBin'] = pd.cut(df['Age'], bins = age_bins, labels = ['Child', 'Teen', 'Adult', 'Middle', 'Senior'])
    df['FareBin'] = pd.cut(df['Fare'], bins = fare_bins, labels = ['Low', 'Mid', 'High', 'VHigh'])

In [140]:
'''
6. Кодирование категориальных признаков

Используем one-hot encoding. Чтобы набор колонок в train и test гарантированно совпадал, объединяем датасеты, кодируем вместе и разделяем обратно
'''

'\n6. Кодирование категориальных признаков\n\nИспользуем one-hot encoding. Чтобы набор колонок в train и test гарантированно совпадал, объединяем датасеты, кодируем вместе и разделяем обратно\n'

In [141]:
n_train = len(train)
combined = pd.concat([train, test], ignore_index = True)
combined = pd.get_dummies(combined, dtype = int)

train = combined.iloc[:n_train].reset_index(drop = True)
test = combined.iloc[n_train:].reset_index(drop = True)

In [142]:
'''
7. Взаимодействия признаков

Добавляем произведения ключевых признаков — модель может уловить нелинейные эффекты(например, женщина в 1 классе vs мужчина в 3 классе)
'''

'\n7. Взаимодействия признаков\n\nДобавляем произведения ключевых признаков — модель может уловить нелинейные эффекты(например, женщина в 1 классе vs мужчина в 3 классе)\n'

In [143]:
for df in [train, test]:
    df['Sex_Pclass'] = df['Sex_male'] * df['Pclass']
    df['FamSize_Pclass'] = df['FamilySize'] * df['Pclass']
    df['Age_Pclass'] = df['Age'] * df['Pclass']

In [144]:
'''
8. Формирование X и y

Проверяем, что колонки в train и test совпадают и NaN не осталось
'''

'\n8. Формирование X и y\n\nПроверяем, что колонки в train и test совпадают и NaN не осталось\n'

In [145]:
X = train.drop(columns = ['Survived'])
y = train['Survived']
X_test = test.drop(columns = ['Survived'])

assert list(X.columns) == list(X_test.columns)
assert X.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print(f"X: {X.shape}, X_test: {X_test.shape}")

X: (891, 59), X_test: (418, 59)


In [146]:
'''
9. Подбор гиперпараметров

Используем StratifiedKFold — он сохраняет пропорцию классов в каждом фолде, что важно при дисбалансе. Оптимизируем ROC-AUC: метрика устойчивее accuracy
и меньше шумит при подборе.
'''

'\n9. Подбор гиперпараметров\n\nИспользуем StratifiedKFold — он сохраняет пропорцию классов в каждом фолде, что важно при дисбалансе. Оптимизируем ROC-AUC: метрика устойчивее accuracy\nи меньше шумит при подборе.\n'

In [147]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    'n_estimators': [200, 300, 400, 500],
    'learning_rate': [0.03, 0.05, 0.07, 0.1],
    'max_depth': [2, 3, 4],
    'min_samples_split': [2, 3, 5],
    'subsample': [0.5, 0.8, 1.0],
}

grid = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
)

grid.fit(X, y)

print(f"\nЛучшие параметры: {grid.best_params_}")
print(f"ROC-AUC на CV: {grid.best_score_:.5f}")

Fitting 5 folds for each of 432 candidates, totalling 2160 fits

Лучшие параметры: {'learning_rate': 0.03, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 200, 'subsample': 0.8}
ROC-AUC на CV: 0.88879


In [148]:
'''
10. Усреднение предсказаний по фолдам

Обучаем модель на каждом фолде с лучшими параметрами и усредняем предсказания
Это снижает дисперсию и даёт более стабильный результат, чем одна модель, обученная на всём train.

Попутно получаем OOF-предсказания (out-of-fold) — честную оценку качества
'''

'\n10. Усреднение предсказаний по фолдам\n\nОбучаем модель на каждом фолде с лучшими параметрами и усредняем предсказания\nЭто снижает дисперсию и даёт более стабильный результат, чем одна модель, обученная на всём train.\n\nПопутно получаем OOF-предсказания (out-of-fold) — честную оценку качества\n'

In [149]:
best_params = grid.best_params_

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    model = GradientBoostingClassifier(
        random_state=RANDOM_STATE + fold,
        **best_params,
    )
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])

    oof_preds[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / cv.get_n_splits()

print(f"OOF ROC-AUC: {roc_auc_score(y, oof_preds):.5f}")

OOF ROC-AUC: 0.88300


In [150]:
'''
11. Подбор порога классификации

Стандартный порог 0.5 не всегда оптимален для accuracy. Перебираем значения по OOF-предсказаниям и выбираем тот, что даёт максимальную точность на train
'''

'\n11. Подбор порога классификации\n\nСтандартный порог 0.5 не всегда оптимален для accuracy. Перебираем значения по OOF-предсказаниям и выбираем тот, что даёт максимальную точность на train\n'

In [151]:
thresholds = np.arange(0.30, 0.71, 0.01)
best_thr, best_acc = 0.5, 0

for t in thresholds:
    acc = accuracy_score(y, (oof_preds > t).astype(int))
    if acc > best_acc:
        best_acc, best_thr = acc, t

print(f"Оптимальный порог: {best_thr:.2f}")
print(f"OOF accuracy: {best_acc:.5f}")

Оптимальный порог: 0.53
OOF accuracy: 0.84175


In [152]:
'''
12. Финальные предсказания
'''

'\n12. Финальные предсказания\n'

In [153]:
final_preds = (test_preds > best_thr).astype(int)

print(f"Распределение предсказаний: {pd.Series(final_preds).value_counts().to_dict()}")
print(f"Всего предсказаний: {len(final_preds)}")

Распределение предсказаний: {0: 268, 1: 150}
Всего предсказаний: 418


In [154]:
'''
13. Submission

Kaggle ожидает CSV с двумя колонками: PassengerId (int) и Survived (0/1)
'''

'\n13. Submission\n\nKaggle ожидает CSV с двумя колонками: PassengerId (int) и Survived (0/1)\n'

In [155]:
submission = pd.DataFrame({
    'PassengerId': test_ids.astype('int64'),
    'Survived': final_preds.astype('int64'),
})

assert len(submission) == 418
assert submission['Survived'].isin([0, 1]).all()
assert submission['PassengerId'].is_unique
assert pd.api.types.is_integer_dtype(submission['Survived'])

submission.to_csv('submission.csv', index=False)

print("submission.csv сохранён")
print(submission.head())

submission.csv сохранён
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
